In [ ]:
import numpy as np
import random

In [ ]:
class PolytopiaEnv:
  def __init__(self, grid_size):
    self.grid_size = grid_size;
    self.reset();

  def reset(self):
    self.ally_units = [{"R": 0, "C": 0, "HP":10, "MaxHP":10,"Attack":2,
                        "Defence":2, "Movement": 1, "Range": 1, "Promote":0,
                        "Skills": ["Dash","Fortify"],"Fortified": True}]
    self.ally_cities = [{"R": 0, "C": 0,"Stars":2, "Level":1,
                         "Wall":False,"Population":1,"Prog":0, "Border":3}]
    self.ally_stars = 5

    self.enemy_units = [{"R": 0, "C": 0, "HP":10, "MaxHP":10,"Attack":2,
                        "Defence":2, "Movement": 1, "Range": 1, "Promote":0,
                        "Skills": ["Dash","Fortify"],"Fortified": True}]
    self.enemy_cities = [{"R": 0, "C": 0,"Stars":2, "Level":1,
                         "Wall":False,"Population":1,"Prog":0, "Border":3}]
    self.enemy_stars = 5

    self.generate_terrain()
    self.generate_resources()
    self.generate_villages()
    self.generate_ruins()

  # generate the terrain on the map
  def generate_terrain(self):
    # 0 = field, 1 = forest, 2 = mountain
    """
    self.terrain = np.random.choice(
        [0,1,2],
        size = (self.grid_size,self.grid_size),
        p = [0.48,0.38,0.14])
    """
    # NO MOUNTAIN
    self.terrain = np.random.choice(
        [0,1],
        size = (self.grid_size,self.grid_size),
        p = [0.48,0.38])

  # generate resources on the map
  def generate_resources(self):
    # resource types: 0 = empty, 1 = fruit, 2 = crop
    #                 3 = animal, 4 = metal

    # make 2D array filled with zeros for resources
    self.resource = np.zeroes(self.grid_size,self.grid_size)

    for R in range(self.grid_size):
      for C in range(self.grid_size):
        spawn = random.random()
        if (self.terrain[R][C] == 0):
          if (spawn <= 0.375):
            self.resource[R][C] = 1
          elif (0.375 < spawn <= 0.75):
            self.resource[R][C] = 2
          else:
            self.resource[R][C] = 0

        elif (self.terrain[R][C] == 1):
          if (spawn <= 0.5):
            self.resource[R][C] = 3
          else:
            self.resource[R][C] = 0
        """
        NO MOUNTAINS
        elif (self.terrain[R][C] == 2):
          if (spawn <= 0.785):
            self.resource[R][C] = 4
          else:
            self.resource[R][C] = 0
        """

  # generate villages on map
  def generate_villages(self):
    # manually placed. 0 = no village, 1 = village, 2 = capital
    # assumed map is 11 x 11 (for now, fix later)
    firstcapH = 0
    firstcapW = 0
    secondcapH = 0
    secondcapW = 0
    numvil = 0
    self.villages = np.zeroes(self.grid_size,self.grid_size)
    for R in range(self.grid_size):
      for C in range(self.grid_size):
        place_vil = self.villages[R][C]
        if (self.terrain[R][C] == 1 and
            self.resource[R][C] == 0 and
            (R+1) != (self.grid_size - 1) and
            (C+1) != (self.grid_size - 1) and
            not self.next_to_vil(R,C)):
            self.villages[R][C] = 1
            numvil+=1
            if (numvil == 1):
              firstcapH = R
              firstcapW = C
            if (numvil >= 2):
              secondcapH = R
              secondcapW = C
    if (numvil >= 2):
      self.villages[firstcapH][firstcapW] = 2
      self.ally_units[0]["R"] = firstcapH
      self.ally_units[0]["C"] = firstcapW
      self.ally_cities[0]["R"] = firstcapH
      self.ally_cities[0]["C"] = firstcapW

      self.villages[secondcapH][secondcapW]= 2
      self.enemy_units[0]["R"] = firstcapH
      self.enemy_units[0]["C"] = firstcapW
      self.enemy_cities[0]["R"] = firstcapH
      self.enemy_cities[0]["C"] = firstcapW
    else:
      # make a predefined configuration
      self.villages[1][4] = 1
      self.villages[3][1] = 1
      self.villages[4][4] = 1
      self.villages[8][9] = 1
      self.villages[9][5] = 1
      self.villages[5][7] = 2
      self.villages[7][2] = 2

      self.ally_units[0]["R"] = 5
      self.ally_units[0]["C"] = 7
      self.ally_cities[0]["R"] = 5
      self.ally_cities[0]["C"] = 7

      self.enemy_units[0]["R"] = 7
      self.enemy_units[0]["C"] = 2
      self.enemy_cities[0]["R"] = 7
      self.enemy_cities[0]["C"] = 2

  # generate ruins on map
  def generate_ruins(self):
    # 0 = no ruin, 1 = ruin
    self.ruins = np.zeroes(self.grid_size,self.grid_size)
    for R in range(self.grid_size):
      for C in range(self.grid_size):
        ruin_spawn = random.random()
        if (ruin_spawn <= 0.025 and
           (self.villages[R][C] != 1 or self.villages[R][C] != 2)):
           self.ruins[R][C] = 1

  # determine if village is next to another
  def next_to_vil(self, R, C):
    for i in range(1,4):
      if (self.village[R+i][C+i] != (1 or 2) and
          self.village[R+i][C] != (1 or 2) and
          self.village[R][C+i] != (1 or 2) and
          self.village[R-i][C-i] != (1 or 2) and
          self.village[R-i][C] != (1 or 2) and
          self.village[R][C-i] != (1 or 2) and
          self.village[R-i][C+i] != (1 or 2) and
          self.village[R+i][C-i] != (1 or 2)):
          return False
    return True

  # attack an enemy unit
  def attack(self, unit_ind, enemy_ind):
    attack = self.ally_units[unit_ind]["Attack"]
    health = self.ally_units[unit_ind]["HP"]
    maxhealth = self.ally_units[unit_ind]["MaxHP"]

    enemydefense = self.enemy_units[enemy_ind]["Defense"]
    enemyhp = self.enemy_units[enemy_ind]["HP"]
    enemymaxhp = self.enemy_units[enemy_ind]["MaxHP"]
    enemyfortified = self.enemy_units[enemy_ind]["Fortified"]

    attackforce =  attack * (health / maxhealth)
    defenseforce = enemydefense * (enemyhp/enemymaxhp)

    if enemyfortified:
      defenseforce *= 1.5

    totalDamage = attackforce + defenseforce
    attackResult = (attackforce / totalDamage) * attack * 4.5
    defenseResult = (defenseforce / totalDamage) * enemydefense * 4.5

    self.ally_units[unit_ind]["HP"] -= defenseResult
    self.enemy_units[enemy_ind]["HP"] -= attackResult

    if (self.ally_units[unit_ind]["HP"] <= 0):
      self.ally_units.pop(unit_ind)
      if (self.enemy_units[enemy_ind]["HP"] <= 0):
        self.enemy_units.pop(unit_ind)
      else:
        self.enem_units[unit_ind]["Promote"] += 1
    elif (self.enemy_units[enemy_ind]["HP"] <= 0):
      self.ally_units[unit_ind]["Promote"] += 1

  # collect a resource
  def collect_resource(self, R, C, city_ind, type):
    # resource types: 0 = empty, 1 = fruit, 2 = crop
    #                 3 = animal, 4 = metal
    if (type == 1):
      if (self.ally_stars < 2):
        return False
      else:
        self.ally_stars -= 2
        self.resources[R][C] = 0
        self.ally_cities[city_ind]["Prog"] += 1
        return True
    elif (type == 2):
      if (self.ally_stars < 5):
        return False
      else:
        self.ally_stars -= 5
        self.resources[R][C] = 0
        self.ally_cities[city_ind]["Prog"] += 2
        return True
    elif(type == 3):
      if (self.ally_stars < 2):
        return False
      else:
        self.ally_stars -= 2
        self.resources[R][C] = 0
        self.ally_cities[city_ind]["Prog"] += 1
        return True
    """
    elif(type == 4):
      if (self.ally_stars < 5):
        return False
      else:
        self.ally_stars -= 5
        self.resources[R][C] = 0
        self.ally_cities[city_ind]["Prog"] += 2
        return True
    """

  # move a unit to a target location
  def move(self, unit_ind, targ_R, targ_C):
    dist = self.ally_units[unit_ind]["Movement"]
    R = self.ally_units[unit_ind]["R"]
    C = self.ally_units[unit_ind]["C"]
    if (abs(targ_R - R) <= dist and abs(targ_C - C) <= dist):
      self.ally_units[unit_ind]["R"] = targ_R
      self.ally_units[unit_ind]["C"] = targ_C
      return True
    else:
      return False

  # train a unit
  def train(self, unit):
    if unit == "Warrior":
      if (self.ally_stars >= 2):
        self.ally_units.append({"R": 0, "C": 0, "HP":10, "MaxHP":10,"Attack":2,
                          "Defence":2, "Movement": 1, "Range": 1, "Promote":0,
                          "Skills": ["Dash","Fortify"],"Fortified": True})
        self.ally_stars -=2
        return True
      else:
        return False

  # don't know how to implement either of these 2 because of the turn system
  def capture(self, unit_ind, city_ind):
    pass
  def collect_ruin(self, unit_ind, city_ind):
    pass